# 03 — Promotion Analysis

This notebook loads promotion performance metrics from `src.promotion_metrics` and
analyzes promotional lift across 4,910 beverage SKUs.  The two-signal promo flag
(causal display/mailer + discount-field fallback) is validated, the lift index
distribution is visualized, and the top-20 over-performing SKUs are identified
for prioritized promotional investment.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
print('Ready')

In [ ]:
from src.data_cleaning import PROCESSED_DIR

pp = pd.read_parquet(PROCESSED_DIR / 'promotion_performance.parquet')
print(f'Promotion performance rows  : {len(pp):,}')
print(f'Columns                     : {list(pp.columns)}')
print()
print('Products with valid lift_index (noise-filtered: >=50 units, >=3 promo weeks, >=3 non-promo weeks):')
pp_filt = pp[
    (pp['units_sold'] >= 50) &
    (pp['promo_weeks'] >= 3) &
    (pp['non_promo_weeks'] >= 3) &
    pp['lift_index_vs_category'].notna()
].copy()
print(f'  Filtered SKUs: {len(pp_filt):,}')
print(f'  SKUs >= 2.4x lift index : {(pp_filt["lift_index_vs_category"] >= 2.4).sum()}')
print(f'  SKUs >= 2.0x lift index : {(pp_filt["lift_index_vs_category"] >= 2.0).sum()}')
print(f'  Mean lift_index         : {pp_filt["lift_index_vs_category"].mean():.3f}')
print(f'  Median lift_index       : {pp_filt["lift_index_vs_category"].median():.3f}')

In [ ]:
# Top 20 products by lift_index_vs_category
bp = pd.read_parquet(PROCESSED_DIR / 'beverage_products.parquet')
top20 = pp_filt.nlargest(20, 'lift_index_vs_category')[[
    'product_id', 'beverage_category', 'lift_index_vs_category',
    'promotion_lift', 'category_avg_lift', 'units_sold', 'promo_weeks', 'non_promo_weeks'
]].merge(bp[['product_id', 'commodity_desc']], on='product_id', how='left')

print('Top 20 products by lift index vs. category average:')
print(top20.to_string(index=False))

In [ ]:
# Promo signal statistics
bt = pd.read_parquet(PROCESSED_DIR / 'beverage_transactions.parquet')
n_total = len(bt)
n_promo = bt['is_promo'].sum()
n_causal = bt['causal_promo'].sum()
n_discount = ((bt['retail_disc'] < 0) | (bt['coupon_disc'] < 0) | (bt['coupon_match_disc'] < 0)).sum()

print(f'Total beverage transaction lines : {n_total:,}')
print(f'is_promo = True                  : {n_promo:,} ({n_promo/n_total*100:.1f}%)')
print(f'  Signal A (causal only)         : {n_causal:,} ({n_causal/n_total*100:.1f}%)')
print(f'  Signal B (discount only)       : {n_discount:,} ({n_discount/n_total*100:.1f}%)')
print()
print('Promo revenue share by category:')
promo_cat = bt.groupby('beverage_category').apply(
    lambda g: pd.Series({
        'promo_rev_pct': g.loc[g['is_promo'], 'sales_value'].sum() / g['sales_value'].sum() * 100
    })
).reset_index()
print(promo_cat.to_string(index=False))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pathlib

CHARTS = pathlib.Path('..') / 'outputs' / 'charts'

# Chart 1: Lift index distribution
lift_vals = pp_filt['lift_index_vs_category'].clip(0, 5)
fig, ax = plt.subplots(figsize=(9, 5))
n, bins, patches = ax.hist(lift_vals, bins=50, color='#ff7f0e', edgecolor='white', linewidth=0.4)
ax.axvline(1.0, color='#1f77b4', linewidth=1.8, linestyle='--', label='1.0 (category avg)')
ax.axvline(2.4, color='#d62728', linewidth=1.8, linestyle='--', label='2.4x threshold (36 SKUs)')
ax.set_title('Lift Index Distribution (noise-filtered, clipped at 5x)', fontsize=11)
ax.set_xlabel('Lift Index vs. Category Average', fontsize=10)
ax.set_ylabel('Number of SKUs', fontsize=10)
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(CHARTS / 'lift_distribution.png', dpi=130)
plt.close(fig)
print('Saved lift_distribution.png')

# Chart 2: Top 20 lift (horizontal bar)
top20_plot = top20.sort_values('lift_index_vs_category', ascending=True)
cats = top20_plot['beverage_category'].unique()
cmap = plt.colormaps['tab10']
cat_colors = {cat: cmap(i / max(len(cats)-1, 1)) for i, cat in enumerate(cats)}
bar_colors = [cat_colors[c] for c in top20_plot['beverage_category']]

labels = top20_plot['product_id'].astype(str) + ' - ' + top20_plot['commodity_desc'].fillna('').str.title()

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(labels, top20_plot['lift_index_vs_category'], color=bar_colors, edgecolor='white')
ax.axvline(2.4, color='#d62728', linewidth=1.5, linestyle='--')
for bar, val in zip(bars, top20_plot['lift_index_vs_category']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.2f}x', va='center', fontsize=8)
legend_patches = [mpatches.Patch(color=c, label=cat) for cat, c in cat_colors.items()]
ax.legend(handles=legend_patches, fontsize=8)
ax.set_title('Top 20 Products by Lift Index vs. Category Average', fontsize=11)
ax.set_xlabel('Lift Index vs. Category Average', fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(CHARTS / 'top20_lift.png', dpi=130)
plt.close(fig)
print('Saved top20_lift.png')

## Summary — Promotion Analysis

- **62.0% of beverage transaction lines** are flagged as promotional; promo revenue share is **63.6%**.
- Dual-signal promo flag: Signal A (causal display/mailer from `causal_data.csv`) covers ~6.5% of
  beverage (product, store, week) combos; Signal B (any discount field < 0) carries the majority.
- **873 SKUs** pass the noise filter (>=50 units, >=3 promo weeks, >=3 non-promo weeks).
- Mean promotion lift (unit-weighted): **1.94x**; Median lift: **0.948x**.
  The median < 1 means the *typical* promoted SKU does not accelerate velocity — lift is
  concentrated in a minority of high-performers.
- **36 SKUs deliver >= 2.4x category-average lift**; 61 deliver >= 2.0x.
- Top performer: **product 1016982** (Carbonated Soft Drinks) — **14.34x category-average lift**.
- Key RGM implication: promotional investment should be concentrated on these 36 high-lift SKUs,
  not distributed uniformly across the category.